# Crash Test of Divergence

**Reframed from the original always-in / buy-and-hold version.** Comparing
a rare-signal strategy's equity curve against buy & hold conflated two
different questions: "does divergence predict a short-term move" and "is
holding whatever position you last got, for however long until the next
signal, a good standalone strategy." With only 10-22 signals per run, most
of what got scored was the second question, dominated by long stale
holding periods, not the signal itself.

This version drops the position/equity-curve backtest entirely and runs a
forward-return event study instead: for every confirmed divergence signal,
what actually happened to price over the following 5/10/20 trading days,
compared to nothing more than "did it move the predicted direction." The
universe is also expanded (3 major FX pairs added) specifically to get a
large enough signal count for the results to mean something.

## Data

In [1]:
import os

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import requests
import requests_cache
from plotly.subplots import make_subplots

requests_cache.install_cache("eodhd_cache", expire_after=60 * 60 * 24)

api_token = os.environ.get("EODHD_API_TOKEN")


def get_ohlc(symbol, from_date, to_date):
    url = f"https://eodhd.com/api/eod/{symbol}"
    query = {
        "api_token": api_token,
        "from": from_date,
        "to": to_date,
        "period": "d",
        "fmt": "json",
    }
    data = requests.get(url, params=query).json()

    df = pd.DataFrame(data)
    df["date"] = pd.to_datetime(df["date"])
    df.sort_values(by="date", inplace=True)
    df.set_index("date", inplace=True)

    # Forex EOD has no corporate actions, so EODHD doesn't return an
    # adjusted_close for FX symbols - close is already the "adjusted" series.
    # NOTE: assumption, not yet verified against a real forex response -
    # confirm on first run and adjust if the field shows up after all.
    if "adjusted_close" not in df.columns:
        df["adjusted_close"] = df["close"]

    # Put highs/lows on the same (adjusted) basis as adjusted_close, so the
    # oscillators that use them aren't mixing two different price series.
    adj_factor = df["adjusted_close"] / df["close"]
    df["adj_high"] = df["high"] * adj_factor
    df["adj_low"] = df["low"] * adj_factor

    # Spot FX has no real trade volume - MFI is skipped for these symbols
    # (see oscillators_for below) but keep the column present so nothing
    # downstream breaks on a missing key.
    if "volume" not in df.columns:
        df["volume"] = np.nan

    df["pct_change"] = df["adjusted_close"].pct_change()
    return df


list_of_assets = [
    "SPY.US", "GLD.US", "IWM.US", "BTC-USD.CC", "EMSG.US", "EXW1.XETRA",
    "EURUSD.FOREX", "GBPUSD.FOREX", "USDJPY.FOREX",
]
FOREX_ASSETS = {"EURUSD.FOREX", "GBPUSD.FOREX", "USDJPY.FOREX"}


## Meet the oscillators

In [2]:
def RSI(df, period=14):
    delta = df["adjusted_close"].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    # Wilder's smoothing
    avg_gain = gain.ewm(alpha=1 / period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / period, adjust=False).mean()
    rs = avg_gain / avg_loss
    df["RSI"] = 100 - (100 / (1 + rs))
    return df, "RSI"


def MACD(df, fast=12, slow=26, signal=9):
    ema_fast = df["adjusted_close"].ewm(span=fast, adjust=False).mean()
    ema_slow = df["adjusted_close"].ewm(span=slow, adjust=False).mean()
    # The MACD line itself - not the histogram, not the signal line.
    df["MACD"] = ema_fast - ema_slow
    return df, "MACD"


def STOCH(df, period=14, smooth=3):
    low_n = df["adj_low"].rolling(period).min()
    high_n = df["adj_high"].rolling(period).max()
    fast_k = 100 * (df["adjusted_close"] - low_n) / (high_n - low_n)
    df["STOCH"] = fast_k.rolling(smooth).mean()
    return df, "STOCH"


def CCI(df, period=20):
    tp = (df["adj_high"] + df["adj_low"] + df["adjusted_close"]) / 3
    mean_dev = tp.rolling(period).apply(lambda x: np.abs(x - x.mean()).mean(), raw=True)
    df["CCI"] = (tp - tp.rolling(period).mean()) / (0.015 * mean_dev)
    return df, "CCI"


def WILLR(df, period=14):
    high_n = df["adj_high"].rolling(period).max()
    low_n = df["adj_low"].rolling(period).min()
    df["WILLR"] = -100 * (high_n - df["adjusted_close"]) / (high_n - low_n)
    return df, "WILLR"


def MFI(df, period=14):
    tp = (df["adj_high"] + df["adj_low"] + df["adjusted_close"]) / 3
    raw_money_flow = tp * df["volume"]
    positive_flow = raw_money_flow.where(tp > tp.shift(1), 0.0)
    negative_flow = raw_money_flow.where(tp < tp.shift(1), 0.0)
    money_ratio = (
        positive_flow.rolling(period).sum() / negative_flow.rolling(period).sum()
    )
    df["MFI"] = 100 - (100 / (1 + money_ratio))
    return df, "MFI"


list_of_oscillators = [RSI, MACD, STOCH, CCI, WILLR, MFI]
osc_by_name = {f.__name__: f for f in list_of_oscillators}

# Pretty names for the charts
osc_labels = {
    "RSI": "RSI(14)",
    "MACD": "MACD(12,26)",
    "STOCH": "Stochastic %K(14,3)",
    "CCI": "CCI(20)",
    "WILLR": "Williams %R(14)",
    "MFI": "MFI(14)",
}


def oscillators_for(asset):
    """MFI needs real trade volume, which spot FX doesn't have - skip it there."""
    if asset in FOREX_ASSETS:
        return [f for f in list_of_oscillators if f.__name__ != "MFI"]
    return list_of_oscillators


## Divergence detection

In [3]:
def find_swings(series, window):
    """Boolean masks marking local troughs and peaks over +/- `window` bars."""
    span = 2 * window + 1
    roll_min = series.rolling(span, center=True).min()
    roll_max = series.rolling(span, center=True).max()
    return series.eq(roll_min), series.eq(roll_max)


def find_divergence_signals(df, price_col, osc_col, window):
    """+1 on a confirmed bullish divergence, -1 on bearish, 0 otherwise.

    Signals are placed at the bar the swing is *confirmed* (window bars after
    the extreme itself), never at the extreme.
    """
    price = df[price_col]
    osc = df[osc_col]
    is_trough, is_peak = find_swings(price, window)
    usable = osc.notna()

    signal = pd.Series(0, index=df.index, dtype=int)
    n = len(df)

    checks = (
        # mask,                lower low / higher high, osc goes the other way, signal
        (is_trough & usable, np.less, np.greater, 1),
        (is_peak & usable, np.greater, np.less, -1),
    )
    for mask, price_cmp, osc_cmp, value in checks:
        swings = np.flatnonzero(mask.to_numpy())
        for prev, cur in zip(swings[:-1], swings[1:]):
            diverges = price_cmp(price.iloc[cur], price.iloc[prev]) and osc_cmp(
                osc.iloc[cur], osc.iloc[prev]
            )
            if not diverges:
                continue
            confirmed_at = cur + window
            if confirmed_at < n:
                signal.iloc[confirmed_at] = value

    return signal


## Forward-return event study

In [4]:
FORWARD_HORIZONS = [5, 10, 20]  # trading days after signal confirmation


def forward_returns_for_signals(df, signal, price_col, horizons):
    """One row per (signal, horizon). `signed_fwd_ret` flips sign on bearish
    signals, so a "correct" call - price moving the predicted direction - is
    always positive, whichever way the signal itself pointed. Both dates used
    (confirmation bar and confirmation+horizon) are at or after the bar the
    signal already fires on, so this stays free of lookahead the same way the
    signal placement itself already is.
    """
    price = df[price_col]
    n = len(df)
    rows = []
    for idx in np.flatnonzero(signal.to_numpy()):
        direction = int(signal.iloc[idx])  # +1 bull, -1 bear
        for h in horizons:
            target = idx + h
            if target >= n:
                continue
            raw_fwd_ret = price.iloc[target] / price.iloc[idx] - 1
            rows.append(
                {
                    "date": df.index[idx],
                    "horizon": h,
                    "direction": direction,
                    "raw_fwd_ret": raw_fwd_ret,
                    "signed_fwd_ret": raw_fwd_ret * direction,
                }
            )
    return rows


windows = [5, 10, 15, 20, 25, 30, 40, 50]
event_records = []

for asset in list_of_assets:
    df_working = get_ohlc(asset, "2020-01-01", "2025-12-31")

    for osc_function in oscillators_for(asset):
        df_working, osc_name = osc_function(df_working)

        for window in windows:
            signal = find_divergence_signals(df_working, "adjusted_close", osc_name, window)
            if (signal != 0).sum() == 0:
                continue

            for row in forward_returns_for_signals(
                df_working, signal, "adjusted_close", FORWARD_HORIZONS
            ):
                row.update({"asset": asset, "osc_name": osc_name, "window": window})
                event_records.append(row)

events_df = pd.DataFrame(event_records)

n_distinct_signals = events_df.drop_duplicates(
    subset=["asset", "osc_name", "window", "date"]
).shape[0]
print(f"Total distinct divergence signals across the universe: {n_distinct_signals}")
print(f"Total (signal, horizon) observations: {len(events_df)}")
events_df


Total distinct divergence signals across the universe: 4626
Total (signal, horizon) observations: 13825


,date,horizon,direction,raw_fwd_ret,signed_fwd_ret,asset,osc_name,window
0,2020-03-30,5,1,0.012268,0.012268,SPY.US,RSI,5
1,2020-03-30,10,1,0.084617,0.084617,SPY.US,RSI,5
2,2020-03-30,20,1,0.092031,0.092031,SPY.US,RSI,5
3,2020-12-24,5,-1,-0.000569,0.000569,SPY.US,RSI,5
4,2020-12-24,10,-1,0.026260,-0.026260,SPY.US,RSI,5
...,...,...,...,...,...,...,...,...
13820,2024-01-11,10,-1,0.019866,-0.019866,USDJPY.FOREX,WILLR,50
13821,2024-01-11,20,-1,0.022128,-0.022128,USDJPY.FOREX,WILLR,50
13822,2024-11-07,5,1,0.010844,0.010844,USDJPY.FOREX,WILLR,50
13823,2024-11-07,10,1,0.010485,0.010485,USDJPY.FOREX,WILLR,50


## Aggregation

In [5]:
MIN_SIGNALS_PER_CUT = 10  # applied to the asset/window-bucket breakdowns below,
                          # not to the headline pooled-by-oscillator summary

events_df["osc_label"] = events_df["osc_name"].map(osc_labels)


def window_bucket(w):
    if w <= 15:
        return "Fast (<=15)"
    elif w <= 30:
        return "Medium (16-30)"
    return "Slow (>30)"


events_df["window_bucket"] = events_df["window"].apply(window_bucket)

# Headline result: pooled across every asset and window, per oscillator and horizon.
summary_by_osc_horizon = (
    events_df.groupby(["osc_label", "horizon"])["signed_fwd_ret"]
    .agg(n="size", win_rate=lambda s: (s > 0).mean(), mean="mean", median="median")
    .reset_index()
    .sort_values(["horizon", "mean"], ascending=[True, False])
)
summary_by_osc_horizon


,osc_label,horizon,n,win_rate,mean,median
6,MFI(14),5,566,0.512367,-0.000283,0.001346
15,Williams %R(14),5,996,0.488956,-0.000409,-0.000383
3,"MACD(12,26)",5,640,0.448437,-0.000481,-0.001334
9,RSI(14),5,667,0.470765,-0.000889,-0.000939
0,CCI(20),5,873,0.486827,-0.001311,-0.000413
12,"Stochastic %K(14,3)",5,884,0.480769,-0.001435,-0.000455
16,Williams %R(14),10,992,0.511089,-0.000038,0.000801
4,"MACD(12,26)",10,638,0.456113,-0.001540,-0.002644
1,CCI(20),10,871,0.473020,-0.001613,-0.001285
7,MFI(14),10,565,0.502655,-0.001724,0.000318


## Results

In [6]:
# Chart 1 - mean signed forward return by oscillator, one bar group per horizon
summary_by_osc_horizon["horizon_label"] = summary_by_osc_horizon["horizon"].astype(str)

fig1 = px.bar(
    summary_by_osc_horizon,
    x="mean",
    y="osc_label",
    color="horizon_label",
    orientation="h",
    barmode="group",
    category_orders={"horizon_label": ["5", "10", "20"]},
    title="Mean Signed Forward Return by Oscillator and Horizon (pooled across assets/windows)",
    labels={"mean": "Mean signed forward return", "osc_label": "", "horizon_label": "Days after signal"},
)
fig1.update_layout(height=500, width=850)
fig1.show()


Figure

In [7]:
# Chart 2 - does the ranking hold across assets, or flip? Headline horizon = 10 days.
HEADLINE_HORIZON = 10

by_asset = (
    events_df[events_df["horizon"] == HEADLINE_HORIZON]
    .groupby(["osc_label", "asset"])["signed_fwd_ret"]
    .agg(n="size", mean="mean")
    .reset_index()
)
by_asset = by_asset[by_asset["n"] >= MIN_SIGNALS_PER_CUT]

fig2 = px.density_heatmap(
    by_asset,
    x="asset",
    y="osc_label",
    z="mean",
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0,
    title=f"Mean Signed {HEADLINE_HORIZON}-Day Forward Return by Oscillator and Asset (n >= {MIN_SIGNALS_PER_CUT})",
    labels={"mean": "Mean signed fwd ret", "osc_label": "", "asset": ""},
)
fig2.update_layout(height=450, width=850)
fig2.show()


In [8]:
# Chart 3 - does detector sensitivity matter? Headline horizon = 10 days.
by_bucket = (
    events_df[events_df["horizon"] == HEADLINE_HORIZON]
    .groupby(["osc_label", "window_bucket"])["signed_fwd_ret"]
    .agg(n="size", mean="mean")
    .reset_index()
)
by_bucket = by_bucket[by_bucket["n"] >= MIN_SIGNALS_PER_CUT]

osc_order = by_bucket.groupby("osc_label")["mean"].mean().sort_values().index
by_bucket["osc_label"] = pd.Categorical(by_bucket["osc_label"], categories=osc_order, ordered=True)
by_bucket = by_bucket.sort_values("osc_label")

fig3 = px.bar(
    by_bucket,
    x="mean",
    y="osc_label",
    color="window_bucket",
    orientation="h",
    barmode="group",
    category_orders={"window_bucket": ["Fast (<=15)", "Medium (16-30)", "Slow (>30)"]},
    title=f"Mean Signed {HEADLINE_HORIZON}-Day Forward Return: Oscillators by Swing-Window Bucket (n >= {MIN_SIGNALS_PER_CUT})",
    labels={"mean": "Mean signed fwd ret", "osc_label": "", "window_bucket": "Window"},
)
fig3.update_layout(height=550, width=850)
fig3.show()


## A single signal, up close

In [9]:
def plot_divergence_example(asset, osc_name, window, from_date=None, to_date=None):
    df_plot = get_ohlc(asset, "2020-01-01", "2025-12-31")
    df_plot, _ = osc_by_name[osc_name](df_plot)

    signal = find_divergence_signals(df_plot, "adjusted_close", osc_name, window)
    df_plot["signal"] = signal

    if from_date:
        df_plot = df_plot.loc[from_date:to_date]

    bull = df_plot[df_plot["signal"] == 1]
    bear = df_plot[df_plot["signal"] == -1]

    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        row_heights=[0.6, 0.4],
        vertical_spacing=0.05,
        subplot_titles=("Price", osc_labels[osc_name]),
    )

    fig.add_trace(
        go.Scatter(x=df_plot.index, y=df_plot["adjusted_close"], name="Price"), row=1, col=1
    )
    for frame, label, symbol, colour in (
        (bull, "Bullish divergence", "triangle-up", "green"),
        (bear, "Bearish divergence", "triangle-down", "red"),
    ):
        fig.add_trace(
            go.Scatter(
                x=frame.index, y=frame["adjusted_close"], mode="markers", name=label,
                marker=dict(symbol=symbol, size=11, color=colour),
            ),
            row=1, col=1,
        )
        fig.add_trace(
            go.Scatter(
                x=frame.index, y=frame[osc_name], mode="markers", showlegend=False,
                marker=dict(symbol=symbol, size=9, color=colour),
            ),
            row=2, col=1,
        )
    fig.add_trace(go.Scatter(x=df_plot.index, y=df_plot[osc_name], name=osc_name), row=2, col=1)

    fig.update_layout(
        height=650, width=950,
        title=f"{asset} - {osc_labels[osc_name]} divergence, window={window}",
    )
    fig.show()
    return fig


# Pick one concrete, illustrative example: the single best-performing
# (osc, asset, window) signal at the headline horizon, among cuts that
# cleared MIN_SIGNALS_PER_CUT.
candidate_cuts = (
    events_df[events_df["horizon"] == HEADLINE_HORIZON]
    .groupby(["osc_name", "asset", "window"])["signed_fwd_ret"]
    .agg(n="size", mean="mean")
    .reset_index()
)
candidate_cuts = candidate_cuts[candidate_cuts["n"] >= MIN_SIGNALS_PER_CUT]
best_cut = candidate_cuts.sort_values("mean", ascending=False).iloc[0]
print(best_cut)

fig4 = plot_divergence_example(
    best_cut["asset"], best_cut["osc_name"], int(best_cut["window"])
)

# That one cut's own forward-return numbers, per horizon, for reference in the article
one_cut_by_horizon = events_df[
    (events_df["asset"] == best_cut["asset"])
    & (events_df["osc_name"] == best_cut["osc_name"])
    & (events_df["window"] == best_cut["window"])
].groupby("horizon")["signed_fwd_ret"].agg(n="size", win_rate=lambda s: (s > 0).mean(), mean="mean")
one_cut_by_horizon


osc_name         WILLR
asset       BTC-USD.CC
window              30
n                   11
mean          0.067346
Name: 340, dtype: object


,n,win_rate,mean
horizon,,,
5,12,0.583333,0.035403
10,11,1.000000,0.067346
20,11,0.818182,0.088014


## Export figures

In [ ]:
os.makedirs("figures", exist_ok=True)

fig1.write_image("figures/avg-signed-forward-return-by-oscillator.png", scale=2)
fig2.write_image("figures/signed-forward-return-heatmap.png", scale=2)
fig3.write_image("figures/signed-forward-return-by-window-bucket.png", scale=2)
fig4.write_image("figures/divergence-example.png", scale=2)
